In [ ]:
# label_signatures.py

# Labels every signature in sl4shuur/handwriting-signatures-dataset as
# legible / illegible by comparing TrOCR output to the signer's name.

# Two normalizations are compared at a 90% similarity cutoff:

#   A) "basic"      : lower case, strip spaces, symbols AND digits (original plan)
#   B) "digit_map"  : same, but digits that look like letters are converted first
#                     ("13" -> "b", 0 -> o, 1 -> l, 5 -> s, 8 -> b)

# OCR runs once and is cached to ocr_raw_predictions.csv, so changing the
# normalization, threshold or plots afterwards takes seconds.
# Delete that file to force a fresh OCR run.

# Needs in the working folder: signatures_raw.xlsx
# Outputs: signatures_labeled.csv, legibility_distribution.png

In [2]:
!pip install transformers datasets rapidfuzz pandas openpyxl numpy matplotlib pillow sentencepiece accelerate
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached rapidfuzz-3.14.6-cp314-cp314-win_amd64.whl.metadata (12 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached sentencepiece-0.2.3-cp314-cp314-win_amd64.whl.metadata (35 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-26.0.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-4.0.2-cp314-cp314-win_amd64.whl.metadata (18 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiohttp-3.14.5-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlis

Looking in indexes: https://download.pytorch.org/whl/cu124


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU visible")

2.15.0+cu134
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, disable_progress_bar
from rapidfuzz.distance import Levenshtein
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [4]:

# ----------------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------------
CKPT = "microsoft/trocr-base-handwritten"
DATASET = "sl4shuur/handwriting-signatures-dataset"
XLSX = "signatures_raw.xlsx"
OCR_CACHE = "ocr_raw_predictions.csv"
OUT_CSV = "signatures_labeled.csv"
OUT_PNG = "legibility_distribution.png"

THRESHOLD = 90          # primary cutoff (similarity %, >= is legible)
EXTRA_THRESHOLDS = (80, 70)   # printed for reference only
BATCH = 32              # lower to 16 if you hit CUDA out-of-memory

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

disable_progress_bar()


In [6]:

# ----------------------------------------------------------------------------
# Normalizations
# ----------------------------------------------------------------------------
def norm_basic(text):
    """Lower case; keep letters a-z only (drops spaces, symbols, digits)."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    return re.sub(r"[^a-z]", "", str(text).lower())


DIGIT_TO_LETTER = {"0": "o", "1": "l", "5": "s", "8": "b"}


def norm_digit_map(text):
    """Convert look-alike digits to letters first, then apply basic rules."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    t = str(text).lower()
    t = t.replace("13", "b")  # a capital B is often read as "13"
    t = "".join(DIGIT_TO_LETTER.get(c, c) for c in t)
    return re.sub(r"[^a-z]", "", t)


NORMALIZERS = {"basic": norm_basic, "digit_map": norm_digit_map}


def similarity(expected_norm, predicted_norm):
    """Returns (edit distance, similarity in percent)."""
    d = Levenshtein.distance(expected_norm, predicted_norm)
    m = max(len(expected_norm), len(predicted_norm))
    return d, (100.0 if m == 0 else 100.0 * (1.0 - d / m))


In [7]:

# ----------------------------------------------------------------------------
# OCR (runs once, cached)
# ----------------------------------------------------------------------------
def run_ocr(n_expected):
    print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
    processor = TrOCRProcessor.from_pretrained(CKPT)
    model = (
        VisionEncoderDecoderModel.from_pretrained(CKPT, torch_dtype=DTYPE)
        .to(DEVICE)
        .eval()
    )
    dataset = load_dataset(DATASET, split="train")
    print("Images in dataset:", len(dataset))
    if len(dataset) != n_expected:
        raise RuntimeError(f"Dataset has {len(dataset)} rows, spreadsheet has {n_expected}.")

    texts = list(dataset["text"])
    preds = []
    for start in range(0, len(dataset), BATCH):
        imgs = dataset[start:start + BATCH]["image"]
        rgb = [im.convert("RGB") if im.mode != "RGB" else im for im in imgs]
        px = processor(images=rgb, return_tensors="pt").pixel_values.to(DEVICE, DTYPE)
        with torch.no_grad():
            ids = model.generate(px, max_new_tokens=50)
        preds.extend(processor.batch_decode(ids, skip_special_tokens=True))
        if (start // BATCH) % 10 == 0:
            print(f"  OCR {min(start + BATCH, len(dataset))}/{len(dataset)}")
    return texts, preds


In [8]:

def load_or_run_ocr(meta):
    if Path(OCR_CACHE).exists():
        cache = pd.read_csv(OCR_CACHE, keep_default_na=False)
        if len(cache) == len(meta) and list(cache["text"]) == list(meta["text"]):
            print(f"Using cached OCR results from {OCR_CACHE}")
            return list(cache["ocr_raw"])
        print("Cache does not match spreadsheet; re-running OCR.")
    texts, preds = run_ocr(len(meta))
    # Guard against the spreadsheet and dataset being in different orders
    if list(meta["text"]) != texts:
        raise RuntimeError(
            "Spreadsheet caption order differs from the dataset order. "
            "Match rows on the caption text instead of by position."
        )
    pd.DataFrame({"text": texts, "ocr_raw": preds}).to_csv(OCR_CACHE, index=False)
    print(f"Saved raw OCR output to {OCR_CACHE}")
    return preds


In [12]:

# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
def main():
    meta = pd.read_excel(XLSX)
    ocr_raw = load_or_run_ocr(meta)

    out = pd.DataFrame({
        "file name": meta["file name"],
        "signer_id": meta["file name"].str.extract(r"^(.+)_\d+\.\w+$")[0],
        "expected_raw": meta["Extracted Name Letters only"],
        "ocr_raw": ocr_raw,
    })

    # Compute similarity under each normalization (same one applied to both sides)
    for key, fn in NORMALIZERS.items():
        exp = out["expected_raw"].map(fn)
        prd = out["ocr_raw"].map(fn)
        res = [similarity(e, p) for e, p in zip(exp, prd)]
        out[f"expected_{key}"] = exp
        out[f"ocr_{key}"] = prd
        out[f"dist_{key}"] = [r[0] for r in res]
        out[f"sim_{key}"] = [r[1] for r in res]
        out[f"legible_{key}_{THRESHOLD}"] = (out[f"sim_{key}"] >= THRESHOLD).astype(int)

    out.to_csv(OUT_CSV, index=False)

    # ---- Summary -----------------------------------------------------------
    n = len(out)
    print(f"\n{'=' * 64}\nRESULTS on {n} signatures (legible = similarity >= cutoff)\n{'=' * 64}")
    print("Empty OCR output:", int((out["ocr_raw"].str.strip() == "").sum()))

    for key in NORMALIZERS:
        sim = out[f"sim_{key}"]
        print(f"\n[{key}]  mean {sim.mean():.1f}%  median {sim.median():.1f}%")
        for t in (THRESHOLD, *EXTRA_THRESHOLDS):
            leg = int((sim >= t).sum())
            tag = "  <-- primary" if t == THRESHOLD else ""
            print(f"  cutoff {t}%: legible {leg} ({leg / n:.1%}), "
                  f"illegible {n - leg} ({(n - leg) / n:.1%}){tag}")

    a = out[f"legible_basic_{THRESHOLD}"]
    b = out[f"legible_digit_map_{THRESHOLD}"]
    print(f"\nLabels that change from basic -> digit_map at {THRESHOLD}%: "
          f"{int((a != b).sum())} "
          f"(illegible->legible: {int(((a == 0) & (b == 1)).sum())}, "
          f"legible->illegible: {int(((a == 1) & (b == 0)).sum())})")

    # ---- Figure ------------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    colors = {"basic": "skyblue", "digit_map": "mediumseagreen"}
    titles = {
        "basic": "A) Basic: lower case, letters only",
        "digit_map": "B) Digit-to-letter mapping + basic",
    }
    for col, key in enumerate(NORMALIZERS):
        sim = out[f"sim_{key}"]
        ax = axes[0, col]
        ax.hist(sim, bins=np.arange(0, 105, 5), color=colors[key], edgecolor="black")
        ax.axvline(sim.mean(), color="red", ls="--", label=f"Mean {sim.mean():.1f}%")
        ax.axvline(sim.median(), color="green", label=f"Median {sim.median():.1f}%")
        ax.axvline(THRESHOLD, color="black", ls=":", label=f"Cutoff {THRESHOLD}%")
        ax.set_title(titles[key])
        ax.set_xlabel("Similarity score (%)")
        ax.set_ylabel("Number of signatures")
        ax.legend()
        ax.grid(True, ls="--", alpha=0.4)

        ax = axes[1, col]
        leg = int((sim >= THRESHOLD).sum())
        bars = ax.bar(["Illegible", "Legible"], [n - leg, leg],
                      color=["lightcoral", colors[key]], edgecolor="black")
        for bar, v in zip(bars, [n - leg, leg]):
            ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v}\n({v / n:.1%})",
                    ha="center", va="bottom")
        ax.set_title(f"Class split at {THRESHOLD}% cutoff")
        ax.set_ylabel("Number of signatures")
        ax.set_ylim(0, n * 1.15)

    fig.suptitle(f"OCR-based legibility labels, {n} signatures (TrOCR)", fontsize=14)
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=200)
    print(f"\nSaved {OUT_CSV} and {OUT_PNG}")

In [14]:


if __name__ == "__main__":
    main()

Device: cuda NVIDIA GeForce RTX 5060 Laptop GPU


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

C:\Users\caitl\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\caitl\.cache\huggingface\hub\models--microsoft--trocr-base-handwritten. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

... (2,586 outputs left out)

95_9717.png: reconstructing file:   0%|          |  0.00B / 7.53kB            

95_9717.png: downloading bytes:           |  0.00B            

metadata.jsonl:   0%|          | 0.00/155k [00:00<?, ?B/s]

Images in dataset: 1289
  OCR 32/1289
  OCR 352/1289
  OCR 672/1289
  OCR 992/1289
  OCR 1289/1289
Saved raw OCR output to ocr_raw_predictions.csv

RESULTS on 1289 signatures (legible = similarity >= cutoff)
Empty OCR output: 0

[basic]  mean 59.0%  median 60.0%
  cutoff 90%: legible 196 (15.2%), illegible 1093 (84.8%)  <-- primary
  cutoff 80%: legible 374 (29.0%), illegible 915 (71.0%)
  cutoff 70%: legible 506 (39.3%), illegible 783 (60.7%)

[digit_map]  mean 59.1%  median 60.0%
  cutoff 90%: legible 196 (15.2%), illegible 1093 (84.8%)  <-- primary
  cutoff 80%: legible 375 (29.1%), illegible 914 (70.9%)
  cutoff 70%: legible 509 (39.5%), illegible 780 (60.5%)

Labels that change from basic -> digit_map at 90%: 2 (illegible->legible: 1, legible->illegible: 1)

Saved signatures_labeled.csv and legibility_distribution.png
